# Laboratorio 3.1 · Fundamentos de Apache Airflow

**Curso:** Pipelines de Datos y ETL con Python · BSG Institute
**Caso integrador:** la tienda. En la 2.2 y la 2.3 corrimos el pipeline a mano, con las 14 cargas de golpe.
Hoy lo corre **Airflow**, un día a la vez, como pasaría en producción: cada mañana a las 06:00 de CDMX.

| Parte | Qué haces | La pregunta |
|---|---|---|
| 1 | Lo que hoy haces a mano | ¿Qué le falta a un `python -m ...`? |
| 2 | Tu primer DAG | ¿Qué es una tarea y qué viaja entre tareas? |
| 3 | En serie y en paralelo | ¿Cuánto ahorra el paralelismo? |
| 4 | ¿Qué fecha es? | `logical_date`, intervalo de datos y `catchup` |
| 5 | La tienda, día por día | 15 corridas sin escribir un ciclo |
| 6 | Llega una carga rota | ¿Qué se publica cuando falla? |
| 7 | Backfill | Reprocesar el pasado sin romper el presente |

Airflow corre aquí mismo, en la máquina de Colab, en un entorno virtual aparte para no chocar con las librerías de Colab.
Todo se maneja desde la terminal (`airflow ...`); la interfaz web es opcional.

## Preparación (2 a 3 minutos: instala Airflow 3.3.2 en su propio entorno)

In [ ]:
import sys
from pathlib import Path
BASE = Path.cwd()
VENV = BASE / "airflow_venv"
PY = f"{sys.version_info.major}.{sys.version_info.minor}"
!pip -q install uv
!uv venv -q --allow-existing "{VENV}" --python "{sys.executable}"
!uv pip install -q --python "{VENV}/bin/python" "apache-airflow==3.3.2" "polars>=1.10" pyarrow numpy "pandera[polars]>=0.20" graphviz --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-3.3.2/constraints-{PY}.txt"
!"{VENV}/bin/airflow" version

In [ ]:
# @title Instalar el paquete del kit (el mismo código que el kit local)
import base64, io, zipfile
KIT = BASE / "kit"
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(KIT)
print("Kit en", KIT, "-> transformacion/ (la 2.2) y orquestacion/dags/ (hoy)")

In [ ]:
# @title Configurar y arrancar Airflow (standalone: API, scheduler, dag-processor y triggerer en un solo proceso)
import json, os, re, subprocess, time, urllib.request
from datetime import datetime
from zoneinfo import ZoneInfo
import polars as pl
from IPython.display import Markdown, display

AF_HOME = BASE / "airflow_home"
CDMX = ZoneInfo("America/Mexico_City")
os.environ.update({
    "AIRFLOW_HOME": str(AF_HOME),
    "AIRFLOW__CORE__DAGS_FOLDER": str(KIT / "orquestacion" / "dags"),
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "AIRFLOW__CORE__EXECUTOR": "LocalExecutor",
    "AIRFLOW__CORE__SIMPLE_AUTH_MANAGER_ALL_ADMINS": "True",   # sin contraseña: SOLO para aprender en local
    "AIRFLOW__CORE__DAGS_ARE_PAUSED_AT_CREATION": "True",
    "AIRFLOW__CORE__PARALLELISM": "4",
    "AIRFLOW__DAG_PROCESSOR__REFRESH_INTERVAL": "5",
    "AIRFLOW__API__WORKERS": "1",
    "AIRFLOW__API__PORT": "8090",                              # 8090 para no chocar con otro servicio en el 8080
    "AIRFLOW__API__BASE_URL": "http://localhost:8090",
    "AIRFLOW__CORE__EXECUTION_API_SERVER_URL": "http://localhost:8090/execution/",
    "NO_COLOR": "1",                                           # salida de la CLI sin colores (más fácil de leer aquí)
    "PATH": f"{VENV}/bin:" + os.environ["PATH"],                # `airflow` y `python` salen del entorno de Airflow
})

def af(*args):
    """Corre un comando de la CLI de Airflow y regresa su salida, sin códigos de color."""
    r = subprocess.run(["airflow", *args], capture_output=True, text=True)
    r.stdout, r.stderr = (re.sub(r"\x1b\[[0-9;]*m", "", x) for x in (r.stdout, r.stderr))
    return r

def af_json(*args):
    """Igual que af(), pero pide la salida en JSON y la convierte en una lista de dicts."""
    for linea in reversed(af(*args, "-o", "json").stdout.splitlines()):
        if linea.startswith("["):
            return json.loads(linea)
    return []

def es_bitacora(linea):
    """¿Es una línea interna de Airflow (empieza con fecha y hora)?"""
    return linea[:4].isdigit() and linea[4:5] == "-"

def _hora(t):
    return datetime.fromisoformat(t).astimezone(CDMX).strftime("%Y-%m-%d %H:%M") if t else None

def corridas(dag_id):
    """Las corridas de un DAG, con fechas en hora de CDMX."""
    filas = []
    for r in af_json("dags", "list-runs", dag_id):
        dur = None
        if r["start_date"] and r["end_date"]:
            dur = round((datetime.fromisoformat(r["end_date"]) - datetime.fromisoformat(r["start_date"])).total_seconds(), 1)
        filas.append({"logical_date": _hora(r["logical_date"]), "run_after": _hora(r["run_after"]),
                      "estado": r["state"], "segundos": dur, "run_id": r["run_id"]})
    if not filas:
        return "Sin corridas"
    return pl.DataFrame(filas).sort("logical_date", nulls_last=True)

def esperar(dag_id, n, limite=420):
    """Espera a que el DAG tenga al menos n corridas terminadas y ninguna en curso."""
    time.sleep(3)
    fin = time.time() + limite
    while time.time() < fin:
        rs = af_json("dags", "list-runs", dag_id)
        hechas = [r for r in rs if r["state"] in ("success", "failed")]
        if len(hechas) >= n and not any(r["state"] in ("queued", "running") for r in rs):
            break
        time.sleep(3)
    else:
        print("Se acabó la espera; revisa airflow.log")
    return corridas(dag_id)

def tareas(dag_id, run_id):
    """Las tareas de una corrida: estado y en qué segundo empezó y terminó cada una."""
    ts = af_json("tasks", "states-for-dag-run", dag_id, run_id)
    for t in ts:                       # las que no corrieron no tienen horario que mostrar
        if t["state"] in ("upstream_failed", "skipped", None):
            t["start_date"] = t["end_date"] = None
    inicios = [datetime.fromisoformat(t["start_date"]) for t in ts if t["start_date"]]
    t0 = min(inicios) if inicios else None
    seg = lambda t: round((datetime.fromisoformat(t) - t0).total_seconds(), 1) if t and t0 else None
    return pl.DataFrame([{"tarea": t["task_id"], "estado": t["state"], "empieza_s": seg(t["start_date"]),
                          "termina_s": seg(t["end_date"])} for t in ts]).sort("empieza_s", nulls_last=True)

def salida(dag_id, run_id, task_id):
    """Lo que imprimió una tarea en su último intento (sin el ruido interno de Airflow)."""
    intentos = sorted((AF_HOME / "logs" / f"dag_id={dag_id}" / f"run_id={run_id}" / f"task_id={task_id}").glob("attempt=*.log"))
    if not intentos:
        print("Sin log todavía"); return
    for linea in intentos[-1].read_text(encoding="utf-8").splitlines():
        try:
            d = json.loads(linea)
        except ValueError:
            continue
        ev, logger = str(d.get("event", "")), d.get("logger", "")
        if logger.startswith("task.std") or ("SubprocessHook" in logger and not ev.startswith(("Tmp dir", "Output:"))):
            print(ev)
        elif d.get("level") == "error":
            det = "; ".join(f"{e.get('exc_type')}: {e.get('exc_value')}" for e in d.get("error_detail", []))
            print(f"ERROR {ev} {det}")

def ver_archivo(ruta, lenguaje="python"):
    display(Markdown(f"`{ruta}`\n```{lenguaje}\n{Path(ruta).read_text(encoding='utf-8')}\n```"))

# Base de metadatos (SQLite) y arranque en segundo plano
subprocess.run(["airflow", "db", "migrate"], capture_output=True)
bitacora = open(BASE / "airflow.log", "w")
proceso = subprocess.Popen(["airflow", "standalone"], stdout=bitacora, stderr=subprocess.STDOUT, cwd=BASE)
for _ in range(90):
    try:
        estado = json.load(urllib.request.urlopen("http://localhost:8090/api/v2/monitor/health", timeout=3))
        if all(v["status"] == "healthy" for v in estado.values()):
            break
    except Exception:
        pass
    time.sleep(2)
time.sleep(8)          # que el dag-processor lea la carpeta de DAGs
print({k: v["status"] for k, v in estado.items()})

In [ ]:
# Bronze de la tienda: las mismas 14 cargas de la 2.1, la 2.2 y la 2.3 (del 08/09 al 21/09)
!cd kit && python -m transformacion.generar_bronze --landing landing
pl.DataFrame([{"dag_id": d["dag_id"], "pausado": d["is_paused"]} for d in af_json("dags", "list")])

In [ ]:
# @title Opcional: abrir la interfaz web de Airflow en otra pestaña (solo en Colab)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8090, anchor_text="Abrir Airflow (puerto 8090)")
except ImportError:
    print("Fuera de Colab: abre http://localhost:8090")

## Parte 1 · Lo que hoy haces a mano

En la 2.2 corrimos el pipeline así. Funciona, pero piensa en lo que **no** hace:
¿quién lo corre mañana a las 06:00?, ¿qué pasa si la carga no ha llegado?, ¿cómo reprocesas la semana pasada?,
¿dónde queda registro de qué corrió, cuánto tardó y por qué falló?

In [ ]:
!cd kit && python -m transformacion.pipeline --landing landing --salida /tmp/a_mano --hasta 2026-09-08 2>/dev/null | head -8

Un **orquestador** no reemplaza ese comando: lo **envuelve**. Le da horario, dependencias, reintentos, historia de
corridas y una forma de reprocesar el pasado. El comando sigue siendo tuyo; por eso en la 2.2 le pusimos códigos de
salida (0 bien, 2 lote rechazado, 3 falla de esquema): son el idioma con el que el orquestador entiende qué pasó.

## Parte 2 · Tu primer DAG

Un **DAG** es un archivo de Python en la carpeta de DAGs. Con la **TaskFlow API**, cada función con `@task` es una
tarea, y llamar a una función con el resultado de otra crea la dependencia.

In [ ]:
ver_archivo(KIT / "orquestacion/dags/hola_taskflow.py")

`airflow dags test` corre una corrida completa **en tu terminal**, sin esperar al scheduler. Es la forma rápida de
probar un DAG mientras lo escribes.

In [ ]:
r = af("dags", "test", "hola_taskflow")
print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if l.strip() and not es_bitacora(l)))
corridas("hola_taskflow")

Fíjate en la `logical_date`: una corrida **manual** toma como fecha lógica el momento en que la disparaste.
Las corridas **programadas** toman la de su horario; eso lo vemos en la parte 4.

**Para pensar.** `extraer` regresó una lista y `transformar` la recibió. Esa lista viajó como **XCom**: Airflow la
guardó en su base de datos. ¿Qué pasaría si `extraer` regresara un DataFrame de 30 000 pedidos?

La regla: por XCom viajan cosas chicas (un conteo, una ruta, un dict). Los datos van a disco o al lago, y por XCom
viaja **dónde** quedaron.

## Parte 3 · En serie y en paralelo

Dos DAGs con las mismas cuatro tareas: cada una revisa una tabla de bronze y tarda 4 segundos.
En `en_serie` están encadenadas; en `en_paralelo` no dependen entre sí. Las dos terminan en `resumen` (fan-in).

In [ ]:
ver_archivo(KIT / "orquestacion/dags/serie_y_paralelo.py")

In [ ]:
for d in ("en_serie", "en_paralelo"):
    af("dags", "unpause", d)
    af("dags", "trigger", d)
serie, paralelo = esperar("en_serie", 1), esperar("en_paralelo", 1)
print("en_serie:   ", serie["segundos"][0], "s")
print("en_paralelo:", paralelo["segundos"][0], "s")

In [ ]:
tareas("en_paralelo", paralelo["run_id"][0])

Las cuatro revisiones empiezan casi al mismo segundo. El límite lo pone el executor: aquí
`AIRFLOW__CORE__PARALLELISM=4`, así que caben las cuatro. Con `PARALLELISM=2` irían de dos en dos.

**Para pensar.** Si las cuatro tablas vinieran de la **misma** base de datos de producción, ¿querrías leerlas en paralelo?
(En la 3.2 vemos los *pools*: un límite de lugares por recurso.)

## Parte 4 · ¿Qué fecha es?

Tres DAGs con el mismo horario (`0 6 * * *`, las 06:00 de CDMX), que empiezan el 08/09 y terminan el 12/09.
Ya es el pasado: al despausarlos, Airflow decide qué corridas crear.

In [ ]:
ver_archivo(KIT / "orquestacion/dags/reloj.py")

In [ ]:
for d in ("reloj_disparo", "reloj_intervalo", "reloj_sin_catchup"):
    af("dags", "unpause", d)
esperar("reloj_disparo", 5)
esperar("reloj_intervalo", 5)
time.sleep(10)
for d in ("reloj_disparo", "reloj_intervalo", "reloj_sin_catchup"):
    print(f"\n{d}")
    c = corridas(d)
    print(c if isinstance(c, str) else c.select("logical_date", "run_after", "estado"))

In [ ]:
c = corridas("reloj_intervalo")
salida("reloj_intervalo", c["run_id"][0], "mostrar")
c = corridas("reloj_disparo")
salida("reloj_disparo", c["run_id"][0], "mostrar")

Lo que acabas de ver:

| | `reloj_disparo` (lo normal en Airflow 3) | `reloj_intervalo` (como en Airflow 2) |
|---|---|---|
| La corrida del 08/09 se crea | el 08/09 a las 06:00 | el **09/09** a las 06:00 |
| `logical_date` | 08/09 06:00 | 08/09 06:00 |
| Intervalo de datos | cero: [08/09 06:00, 08/09 06:00) | un día: [08/09 06:00, 09/09 06:00) |
| `ds` | 2026-09-08 | 2026-09-08 |

Y `reloj_sin_catchup` no creó **ninguna**: con `catchup=False`, Airflow no recupera el pasado; solo corre de ahora en
adelante, y su `end_date` ya pasó. Con `catchup=True`, crea una corrida por cada horario que se perdió.

**La idea importante:** cada corrida tiene **su** fecha (`logical_date`, y `ds` como texto). Una tarea bien escrita
procesa **su** fecha, no "hoy". Así, correrla hoy, mañana o en un backfill da lo mismo.

## Parte 5 · La tienda, día por día

El pipeline de la 2.2, orquestado. Cada corrida procesa la carga de su `ds`:
cuatro revisiones en paralelo (¿llegó la carga?), `transformar` (el mismo `python -m transformacion.pipeline`, con
`--hasta {ds}`) y `publicar` (mueve un puntero al último corte, sin retroceder nunca).

In [ ]:
ver_archivo(KIT / "orquestacion/dags/tienda_diaria.py")

In [ ]:
af("dags", "unpause", "tienda_diaria")
tienda = esperar("tienda_diaria", 15)       # catchup: 15 corridas, del 08/09 al 22/09, una a la vez
tienda.select("logical_date", "estado", "segundos")

In [ ]:
def metricas():
    filas = []
    for f in sorted((KIT / "lakehouse" / "metricas").glob("*.json")):
        m = json.loads(f.read_text())
        filas.append({"corte": f.stem, "silver_pedidos": m["silver_pedidos"], "cuarentena": m["cuarentena"],
                      "huella_gold": m["huella_gold"], "modificado": datetime.fromtimestamp(f.stat().st_mtime).strftime("%H:%M:%S")})
    return pl.DataFrame(filas)

antes = metricas()
print("vigente:", (KIT / "lakehouse" / "vigente.txt").read_text())
antes

La corrida del 22/09 falló. ¿Por qué? Mira qué imprimió la tarea que falló:

In [ ]:
ultima = tienda["run_id"][-1]
print(tareas("tienda_diaria", ultima))
salida("tienda_diaria", ultima, "revisar_pedidos_web")

Correcto que falle: la carga del 22 **no ha llegado**. Y como `transformar` depende de las cuatro revisiones, ni siquiera
se intentó (`upstream_failed`). Nadie escribió un `if`: la dependencia lo resolvió.

**Para pensar.** El tablero lee `vigente.txt`. ¿Qué fecha muestra ahora mismo? ¿Es la correcta?

## Parte 6 · Llega una carga rota

Llega por fin la carga del 22/09... con el lote roto de la 2.2 (30 % de los pedidos inválidos).
La reintentamos: `airflow tasks clear` borra el estado de las tareas que fallaron y el scheduler las vuelve a correr.

In [ ]:
!cd kit && python -m transformacion.generar_bronze --landing landing --lote-malo ruidoso
af("tasks", "clear", "tienda_diaria", "-s", "2026-09-22", "-e", "2026-09-23", "--only-failed", "-y")
tienda = esperar("tienda_diaria", 15)
print(tareas("tienda_diaria", ultima))
salida("tienda_diaria", ultima, "transformar")
print("\nvigente:", (KIT / "lakehouse" / "vigente.txt").read_text())

Ahora sí llegaron los archivos, las revisiones pasan y `transformar` corre... y falla con **código 2**: lote rechazado
por calidad. `publicar` no corre, y `vigente.txt` sigue en el 21: el tablero no ve el lote roto. Es la **falla cerrada**
de la 2.2, ahora con el orquestador avisando.

(Corregir el lote es trabajo del proveedor; cuando llegue la versión buena, otro `clear` y listo.)

## Parte 7 · Backfill: reprocesar el pasado

Imagina que corregimos un error en `transformacion/` y hay que reprocesar la primera semana. Eso es un **backfill**:
crear (o recrear) corridas para fechas del pasado. Primero, en seco (`--dry-run`), para ver qué haría:

In [ ]:
r = af("backfill", "create", "--dag-id", "tienda_diaria", "--from-date", "2026-09-08T06:00:00-06:00",
       "--to-date", "2026-09-12T06:00:00-06:00", "--reprocess-behavior", "completed", "--dry-run")
print("Corridas que intentaría (logical_date en hora de CDMX):")
for f in re.findall(r"20\d\d-\d\d-\d\d \d\d:\d\d:\d\d[+-]\d\d:\d\d", r.stdout.split("Runs to be attempted")[-1]):
    print("  ", datetime.fromisoformat(f).astimezone(CDMX).strftime("%Y-%m-%d %H:%M"))
print([l.strip() for l in r.stdout.splitlines() if "max_active_runs" in l][0])

Ojo con las fechas: son **instantes**, no días. Las corridas son a las 06:00 de CDMX (12:00 UTC); si pones
`--to-date 2026-09-12`, eso es la medianoche y la corrida del 12 queda fuera.

`--reprocess-behavior completed` vuelve a correr también los días que ya habían salido bien
(`none` solo crea los que faltan; `failed`, los que fallaron). Y mira `max_active_runs = None`: el backfill no dice
cuántos días a la vez va a correr. Ahora, de verdad:

In [ ]:
af("backfill", "create", "--dag-id", "tienda_diaria", "--from-date", "2026-09-08T06:00:00-06:00",
   "--to-date", "2026-09-12T06:00:00-06:00", "--reprocess-behavior", "completed")
time.sleep(5)
tienda = esperar("tienda_diaria", 15)
despues = metricas()
comparar = antes.join(despues, on="corte", suffix="_despues").select(
    "corte", "huella_gold", "huella_gold_despues", "modificado", "modificado_despues")
print("vigente:", (KIT / "lakehouse" / "vigente.txt").read_text())
comparar.head(7)

In [ ]:
# ¿A qué hora (real) empezó `transformar` en cada día del backfill?
filas = []
for rid in tienda.filter(pl.col("logical_date") <= "2026-09-12 23:59")["run_id"]:
    for t in af_json("tasks", "states-for-dag-run", "tienda_diaria", rid):
        if t["task_id"] == "transformar" and t["start_date"]:
            hora = datetime.fromisoformat(t["start_date"]).astimezone(CDMX).strftime("%H:%M:%S")
            filas.append({"corte": rid[11:21], "transformar_empezo": hora})
pl.DataFrame(filas)

Tres cosas que comprobar:

1. Los archivos del 08 al 12 se **reescribieron** (mira la hora de modificación) y las huellas son **idénticas**:
   el pipeline es idempotente por fecha.
2. `vigente.txt` sigue en el **21**: reprocesar el pasado no movió lo que ve el tablero.
3. Mira las horas: los cinco días **corrieron al mismo tiempo**. El backfill no respeta el `max_active_runs=1` del DAG;
   tiene su propio `--max-active-runs`. Nuestro DAG aguantó porque cada día escribe **su** carpeta y el puntero nunca
   retrocede. Si `transformar` escribiera siempre en `lakehouse/` (como en la 2.2), el gold publicado sería el del día
   que terminara al último.

## Reto (si te sobra tiempo)

1. Agrega a `tienda_diaria.py` una tarea `ventas_del_dia`, después de `transformar`, que lea
   `lakehouse/corte={ds}/gold/ventas_diarias.parquet` e imprima el total del día anterior al corte.
   Edita el archivo en `kit/orquestacion/dags/`, espera unos segundos a que Airflow lo relea y pruébalo con
   `af("dags", "test", "tienda_diaria", "2026-09-15")`.
2. ¿Qué pasa si escribes `from datetime import date` y usas `date.today()` en lugar de `ds`? Piensa qué haría el
   backfill de la parte 7 con esa versión.

In [ ]:
# Tu respuesta del reto 1 aquí

## Al terminar: apaga Airflow

In [ ]:
proceso.terminate()
proceso.wait(timeout=30)
print("Airflow detenido")

## Para llevar

- Un orquestador **envuelve** tus comandos: horario, dependencias, reintentos, historia y backfill.
- TaskFlow: `@dag`, `@task`, y la dependencia sale de pasar resultados. Por XCom viajan cosas chicas.
- Sin dependencias entre tareas, Airflow las corre **en paralelo** (hasta donde lo deje el executor).
- Cada corrida tiene su **fecha lógica**. En Airflow 3 un cron es un **disparo**; el intervalo de un día es opcional.
- `catchup` recupera el pasado; el **backfill** lo reprocesa. Los dos solo son seguros si cada corrida procesa **su**
  fecha y escribe **su** partición.